In [ ]:
import numpy as np
from ome_types import from_xml, to_xml
from ome_types.model import OME, Image, Pixels, Channel, TiffData, Plane, Pixels_DimensionOrder, UnitsLength
from ome_types.model.simple_types import PixelType
import tifffile


import ome_types
from pprint import pprint
from ome_types.model import Plane

# from odemis.util.dataio import open_acquisition
# test-data-03/stripped-metadata/single-channel-stripped-metadata.ome.tiff

import sys

def add_odemis_path():
    """Add the odemis path to the python path"""
    def parse_config(path) -> dict:
        """Parse the odemis config file and return a dict with the config values"""
        
        with open(path) as f:
            config = f.read()

        config = config.split("\n")
        config = [line.split("=") for line in config]
        config = {line[0]: line[1].replace('"', "") for line in config if len(line) == 2}
        return config

    odemis_path = "/etc/odemis.conf"
    config = parse_config(odemis_path)
    sys.path.append(f"{config['DEVPATH']}/odemis/src")  # dev version
    sys.path.append("/usr/lib/python3/dist-packages")   # release version + pyro4

add_odemis_path()

from odemis.util.dataio import open_acquisition
from odemis import model

import tifffile as tff


# support single-channel images, z-stacks, multi-channel images

PATH = "/home/patrick/development/scratch/test-images/test-data-03/z-series.ome.tiff"
image_data = open_acquisition(PATH)

n_channels = len(image_data)

image_data = image_data[0].getData() # for now handle only single channel images
print(image_data.shape)

pixel_size = image_data.metadata[model.MD_PIXEL_SIZE]
pos = image_data.metadata[model.MD_POS]
name = image_data.metadata[model.MD_DESCRIPTION]
dims = image_data.metadata[model.MD_DIMS]
exp_time = image_data.metadata[model.MD_EXP_TIME]

# TODO: channel data

# channels



# tiff_data_blocks
n_channels = 1
n_z_slices = image_data.shape[2]
n_blocks = n_channels * n_z_slices

ifd = 0
channels = []
tiff_data_blocks = []
planes = []
for nc in range(n_channels):


    channels.append(
        Channel(
            id=f"Channel:0:{nc}",
            name = name,
            samples_per_pixel=1
    ))

    for nz in range(n_z_slices):

        # add tiff data block
        tiff_data_blocks.append(
            TiffData(
                plane_count=1,
                ifd=ifd,
                first_z=nz,
                first_c=nc,
                first_t=0
            )
        )

        # add plane
        planes.append(
            Plane(
                the_c=nc,
                the_t=0,
                the_z=nz,
                exposure_time=exp_time,
                position_x=pos[0],
                position_y=pos[1],
                position_z=pos[2],
                position_x_unit=UnitsLength.METER,
                position_y_unit=UnitsLength.METER,
                position_z_unit=UnitsLength.METER,
            )
        )

        ifd += 1



from pprint import pprint 
pprint(planes)
print(tiff_data_blocks)


 # Create OME metadata structure
ome = OME()
image = Image(
    id="Image:0",
    name="Sample Z-stack Image",
    pixels=Pixels(
        id="Pixels:0",
        dimension_order=Pixels_DimensionOrder.XYZCT,
        type=PixelType.UINT16,
        size_x=image_data.shape[-1],
        size_y=image_data.shape[-2],
        size_z=image_data.shape[-3],
        size_c=1,
        size_t=1,
        physical_size_x=pixel_size[0],
        physical_size_y=pixel_size[1],
        physical_size_z=pixel_size[2],
        physical_size_x_unit=UnitsLength.METER,
        physical_size_y_unit=UnitsLength.METER,
        physical_size_z_unit=UnitsLength.METER,
        channels=channels,
        tiff_data_blocks=tiff_data_blocks,
        planes=planes,
    )
)
ome.images.append(image)

# Convert OME object to XML string
ome_xml = to_xml(ome)

# Save the image with OME-XML metadata
with tifffile.TiffWriter('output_z_stack.ome.tiff', bigtiff=True) as tif:
    for z in range(image_data.shape[2]):
        tif.write(image_data[0, 0, z], contiguous=True, metadata={'axes': 'YX'})
    tif.overwrite_description(ome_xml)

print("Z-stack image saved successfully with OME-XML metadata.")













### Working exmaple


In [ ]:











# Create OME metadata structure
ome = OME()
image = Image(
    id="Image:0",
    name="Sample Z-stack Image",
    pixels=Pixels(
        id="Pixels:0",
        dimension_order=Pixels_DimensionOrder.XYZCT,
        type=PixelType.UINT16,
        size_x=1392,
        size_y=1040,
        size_z=3,
        size_c=1,
        size_t=1,
        physical_size_x=0.119047619047619,
        physical_size_y=0.119047619047619,
        physical_size_z=1.5,
        channels=[
            Channel(
                id="Channel:0:0",
                name="Filtered colour 1",
                samples_per_pixel=1
            )
        ],
        tiff_data_blocks=[
            TiffData(plane_count=1, ifd=0, first_z=0, first_c=0, first_t=0),
            TiffData(plane_count=1, ifd=1, first_z=1, first_c=0, first_t=0),
            TiffData(plane_count=1, ifd=2, first_z=2, first_c=0, first_t=0)
        ],
        planes=[
            Plane(
                the_c=0,
                the_t=0,
                the_z=0,
                exposure_time=0.1,
                position_x=48790.8,
                position_y=-23.602647672,
                position_z=1691.5
            ),
            Plane(
                the_c=0,
                the_t=0,
                the_z=1,
                exposure_time=0.1,
                position_x=48790.8,
                position_y=-23.602647672,
                position_z=1693.0
            ),
            Plane(
                the_c=0,
                the_t=0,
                the_z=2,
                exposure_time=0.1,
                position_x=48790.8,
                position_y=-23.602647672,
                position_z=1694.5
            )
        ]
    )
)
ome.images.append(image)

# Convert OME object to XML string
ome_xml = to_xml(ome)

# Save the image with OME-XML metadata
with tifffile.TiffWriter('output_z_stack.ome.tiff', bigtiff=True) as tif:
    for z in range(3):
        tif.write(image_data[z], contiguous=True, metadata={'axes': 'YX'})
    tif.overwrite_description(ome_xml)

print("Z-stack image saved successfully with OME-XML metadata.")